# Giant-Donut Single-Sided Fit — Pupil Comparison

**Author:** Aaron Roodman
**Date Created:** 2026-06-14
**Last Modified:** 2026-06-14
**Status:** In Progress
**Keywords:** giant donut, FAM, defocus, danish, pupil, ISR, single-sided

## Description

Interactive tool to study **how well the optical pupil is modelled**, using the large
(8 mm) defocus FAM images. There are no donut tables or pipeline fits for these, so we:

1. Run **minimal ISR** on the raw (overscan + nominal gains only — no calibs needed).
2. Display a CCD; **click a giant donut** to select it.
3. Cut a large (~1000 px) postage stamp and run a **single-sided Danish fit** with a
   **pure-defocus, Z4-only** model (`startWithIntrinsic=False`), so the model is just
   the optical pupil at the right defocus/size.
4. Show **data / model / residual** + the fitted Z4 — the residual reveals where the
   modelled pupil disagrees with the real donut (e.g. central obscuration).

The defocus comes from the exposure's `observation_reason` label (`intra_8mm` /
`extra_8mm`); it is tunable via `defocus_mm`. The donut size is set by the defocal
offset (physical mm), and the Z4-only fit absorbs the small residual defocus.

**Output:** interactive in-notebook inspector (no files written by default).

**Based on:** `wfs_donut_selection_stages.ipynb`; raw in `LSSTCam/raw/all`
(repo `/repo/main`).

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-06-14 | Aaron Roodman | Initial version: minimal-ISR + click-to-select + single-sided Z4-only Danish fit for giant FAM donuts; data/model/residual pupil comparison |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Interactive Giant-Donut Inspector](#interactive)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

butler_repo = "/repo/main"
raw_collection = "LSSTCam/raw/all"
instrument = "LSSTCam"
cam_name = "LSSTCam"

# Exposure: a labelled FAM defocus frame. observation_reason gives the defocus:
#   20251023: seq 337/338 = extra_8mm, seq 340/341 = intra_8mm (seq 339 = unlabelled "test")
exposure = 2025102300340

# Detector to inspect. The pupil model is most interesting near the field CORNERS,
# so use a near-corner science sensor (clean defocus) or a WFS half-chip:
#   near-corner science: R01_S00, R03_S02, R10_S00, R30_S20, R34_S22, R41_S00, R43_S22
#   WFS half-chips:      R00_SW0/SW1, R04_..., R40_..., R44_...  (donut = defocus ± 1.5 mm)
# R22_S11 (field centre) is the verified default.
detector_name = "R22_S11"

# ---- Fit setup ---------------------------------------------------------
stamp_size = 1000                # postage-stamp size (px); giant donuts are ~450 px
binning = 4                      # danish binning (speed: 1000px @ bin4 -> ~2-5 s)
noll_indices = [4]               # Z4-only (pure defocus) — best for pupil comparison
start_with_intrinsic = False     # pure-defocus pupil model (no off-axis intrinsic terms)
defocus_mm = None                # None -> read from the observation_reason label; else override (e.g. 7.6)
defocal_type = None              # None -> from the label ("intra"/"extra")
band = None                      # None -> from the exposure physical_filter (e.g. r_57 -> "r")
select_tol_px = 600              # click must land within this of a donut centre

# ---- Display ----------------------------------------------------------
cmap = "gray"
dpi = 130


<a id='setup'></a>
## Setup & Imports

In [ ]:
import os
import re
import sys
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from astropy.stats import sigma_clipped_stats
from astropy.visualization import ZScaleInterval

import lsst.geom
from lsst.afw.cameraGeom import FIELD_ANGLE, PIXELS
from lsst.obs.lsst import LsstCam
from lsst.ip.isr import IsrTask, IsrTaskConfig
import lsst.daf.butler as dafButler
from lsst.ts.wep import Image as WepImage
from lsst.ts.wep.estimation import WfEstimator
from lsst.ts.wep.utils import getTaskInstrument

sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting

setup_plotting()
warnings.filterwarnings("ignore")          # silence "no rough zero point" / FFT chatter
camera = LsstCam.getCamera()
det_id = {d.getName(): d.getId() for d in camera}


<a id='functions'></a>
## Helper Functions

In [ ]:
def minimal_isr_config():
    """IsrTask config: overscan + assemble + nominal gains only (no calibs)."""
    cfg = IsrTaskConfig()
    for f in dir(cfg):                       # turn every do* flag off...
        if f.startswith("do") and isinstance(getattr(cfg, f), bool):
            try:
                setattr(cfg, f, False)
            except Exception:
                pass
    cfg.doOverscan = True                    # ...then re-enable the few we want
    cfg.doAssembleCcd = True
    cfg.doApplyGains = True
    cfg.doSaturation = True
    if hasattr(cfg, "usePtcGains"):
        cfg.usePtcGains = False              # nominal camera gains, no PTC needed
    return cfg


_ISR_TASK = None

def run_isr(butler, exposure, detector):
    """Overscan+gain ISR on one raw detector; returns the image array (electrons)."""
    global _ISR_TASK
    if _ISR_TASK is None:
        _ISR_TASK = IsrTask(config=minimal_isr_config())
    raw = butler.get("raw", collections=raw_collection,
                     dataId={"instrument": instrument, "exposure": exposure,
                             "detector": detector})
    return np.asarray(_ISR_TASK.run(raw, camera=camera).exposure.image.array, dtype=float)


def parse_defocus_label(reason):
    """Parse 'intra_8mm' / 'extra_8mm' (etc.) -> (defocal_type, defocus_mm) or (None, None)."""
    m = re.search(r"(intra|extra)_(\d+(?:\.\d+)?)mm", str(reason or ""))
    return (m.group(1), float(m.group(2))) if m else (None, None)


def field_angle_deg(detector, x, y):
    """Field angle (deg) at pixel (x, y) on a detector, from camera geometry."""
    fa = camera[detector].getTransform(PIXELS, FIELD_ANGLE).applyForward(
        lsst.geom.Point2D(x, y))
    return (np.degrees(fa.getX()), np.degrees(fa.getY()))


def single_sided_fit(stamp, field_angle, defocal_type, defocus_m, det_name, band,
                     noll, binning, start_intrinsic):
    """Single-sided Danish fit of one stamp; returns dict(data, model, residual, zk)
    or None if the fit failed (e.g. galsim FFT blow-up on a bad/contaminated stamp)."""
    instr = getTaskInstrument(cam_name, det_name)
    instr.defocalOffset = defocus_m
    wim = WepImage(image=stamp.copy(), fieldAngle=field_angle,
                   defocalType=defocal_type, bandLabel=band)
    wf = WfEstimator(algoName="danish", algoConfig={"binning": binning},
                     instConfig=instr, nollIndices=list(noll),
                     startWithIntrinsic=start_intrinsic, units="um", saveHistory=True)
    zk, _ = wf.estimateZk(wim, None)
    if zk is None or np.any(~np.isfinite(zk)):
        return None
    h = wf.history.get(defocal_type, {})
    if "image" not in h or "model" not in h:
        return None
    d = np.asarray(h["image"]); m = np.asarray(h["model"])
    if not np.all(np.isfinite(m)):
        return None
    return {"data": d, "model": m, "residual": d - m, "zk": np.asarray(zk)}


<a id='data'></a>
## Data Access

In [ ]:
butler = dafButler.Butler(butler_repo)
(exp_rec,) = butler.registry.queryDimensionRecords(
    "exposure", where=f"instrument='{instrument}' and exposure={exposure}")
lab_type, lab_mm = parse_defocus_label(exp_rec.observation_reason)
use_type = defocal_type or lab_type or "intra"
use_mm = defocus_mm or lab_mm or 8.0
use_band = band or str(exp_rec.physical_filter).split("_")[0]
print(f"exposure {exposure}: reason={exp_rec.observation_reason!r}, filter={exp_rec.physical_filter}")
print(f"  -> defocal_type={use_type}, defocus={use_mm} mm, band={use_band}")


<a id='interactive'></a>
## Interactive Giant-Donut Inspector

Click a giant donut in the CCD image. The tool cuts a `stamp_size` stamp, runs the
single-sided Z4-only Danish fit at the labelled defocus, and shows data / model /
residual with the fitted Z4. The model is the pure-defocus optical pupil — so the
residual shows where the modelled pupil disagrees with the donut.

Requires the interactive backend (`ipympl`); run the `%matplotlib widget` cell first.
Tune `defocus_mm` (try ~7.6) to drive the fitted Z4 toward zero for the cleanest
pupil comparison. Pick well-isolated donuts (no neighbours in the stamp).

In [ ]:
%matplotlib widget

In [ ]:
class GiantDonutInspector:
    """Click a giant donut -> single-sided Z4-only Danish fit -> data/model/residual."""

    def __init__(self, exposure, detector_name):
        self.exposure = exposure
        self.det_name = detector_name
        self.det = det_id[detector_name]
        self.img = run_isr(butler, exposure, self.det)
        self.defocal_type = use_type
        self.defocus_m = use_mm * 1e-3
        self.band = use_band
        z = ZScaleInterval().get_limits(self.img[np.isfinite(self.img)])

        self.fig = plt.figure(figsize=(12, 12))
        gs = self.fig.add_gridspec(2, 3, height_ratios=[2.4, 1], hspace=0.18, wspace=0.3)
        self.ax_ccd = self.fig.add_subplot(gs[0, :])
        self.ax_st = {k: self.fig.add_subplot(gs[1, i])
                      for i, k in enumerate(("data", "model", "residual"))}
        self.ax_ccd.imshow(self.img, origin="lower", cmap=cmap, vmin=z[0], vmax=z[1],
                           interpolation="nearest")
        self.ax_ccd.set_title(f"{detector_name} (det {self.det}) exp {exposure} "
                              f"{self.defocal_type}_{use_mm:g}mm — click a donut",
                              fontsize=10)
        (self.marker,) = self.ax_ccd.plot([], [], "+", color="red", ms=16, mew=2)
        self.box = None
        for k, ax in self.ax_st.items():
            ax.set_title(k, fontsize=9); ax.set_xticks([]); ax.set_yticks([])
        self.cid = self.fig.canvas.mpl_connect("button_press_event", self.on_click)

    def on_click(self, event):
        if event.inaxes is not self.ax_ccd or event.xdata is None:
            return
        cx, cy = int(round(event.xdata)), int(round(event.ydata))
        half = stamp_size // 2
        ny, nx = self.img.shape
        if not (half <= cx < nx - half and half <= cy < ny - half):
            print("too close to the edge for a full stamp"); return
        stamp = self.img[cy - half:cy + half, cx - half:cx + half]
        self.marker.set_data([cx], [cy])
        if self.box is not None:
            self.box.remove()
        self.box = plt.Rectangle((cx - half, cy - half), stamp_size, stamp_size,
                                 fill=False, ec="red", lw=1.2)
        self.ax_ccd.add_patch(self.box)
        for ax in self.ax_st.values():
            ax.clear(); ax.set_xticks([]); ax.set_yticks([])

        fa = field_angle_deg(self.det, cx, cy)
        res = single_sided_fit(stamp.astype(float), fa, self.defocal_type,
                               self.defocus_m, self.det_name, self.band,
                               noll_indices, binning, start_with_intrinsic)
        if res is None:
            self.ax_st["model"].set_title("fit failed (try a cleaner/centred donut)",
                                          fontsize=9)
            print(f"({cx},{cy}): fit failed")
        else:
            for k, cm in (("data", "viridis"), ("model", "viridis"),
                          ("residual", "RdBu_r")):
                arr = res[k]
                if k == "residual":
                    v = np.nanpercentile(np.abs(arr), 99)
                    self.ax_st[k].imshow(arr, origin="lower", cmap=cm, vmin=-v, vmax=v)
                else:
                    self.ax_st[k].imshow(arr, origin="lower", cmap=cm)
                self.ax_st[k].set_title(k, fontsize=9)
                self.ax_st[k].set_xticks([]); self.ax_st[k].set_yticks([])
            rr = np.std(res["residual"]) / np.std(res["data"])
            z4 = res["zk"][0] * 1e3
            self.ax_ccd.set_title(
                f"{self.det_name} exp {self.exposure} {self.defocal_type}_{use_mm:g}mm  "
                f"| ({cx},{cy})  fieldAngle=({fa[0]:.2f},{fa[1]:.2f})°  "
                f"Z4={z4:.0f} nm  resid/data={rr:.2f}", fontsize=9)
            print(f"({cx},{cy}) fieldAngle=({fa[0]:.3f},{fa[1]:.3f}) deg  "
                  f"Z4={z4:.0f} nm  resid/data rms={rr:.3f}")
        self.fig.canvas.draw_idle()


inspector = GiantDonutInspector(exposure, detector_name)
